In [ ]:
from TL10 import TL10
from lib import fitVRH

import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from matplotlib import ticker

In [ ]:
plt.rcParams.update({
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 12,
    "text.latex.preamble": r"\usepackage[T1]{fontenc}",
    "text.latex.preamble": r"\usepackage{xcolor}"
})

In [ ]:
def conductivity_ht():
    """Fit conductivity vs temperature for TL10 series"""
    mott = lambda a1, t1, t: a1*t**(-1/2)*np.exp(-(t1/t)**0.25)
    fig = plt.figure(figsize=(3.2, 3), tight_layout=True, dpi=200)
    ax = fig.add_gridspec(1, 1, right=.98, left=0.18, top=0.85, bottom=0.16).subplots()
    for sample in TL10.get_samples(start='TL10_10', stop='TL10_30'):
        t_min, t_max = TL10.ht_range(sample)
        data = TL10.read_conductivity(sample).loc[t_max+1:t_min-1]
        temp, cond = data.index.to_numpy(), data.to_numpy()
        par = TL10.plot_params(sample)
        ax.plot(np.log10(temp), np.log10(cond), 
            par['m'][1:], label=par['label'], mec=par['c'], mfc='w', c=par['c'], ms=6)
        # fitting
        a1, t1, *_ = fitVRH(temp, cond, 0.25)
        cond_theory = mott(a1, t1, temp)
        ax.plot(np.log10(temp), np.log10(cond_theory), '--', c=par['c'])
        # print(f'Fitting results [{i}]: {coef}')
    ax.set_ylabel(r'$\log_{10}\sigma$, where $\sigma$ [($\Omega$cm)$^{-1}$]')
    ax.set_xlabel(r'$\log_{10}T$, where $T$ [K]')
    ax.set_ylim(-8, 0)
    ax.set_xlim(np.log10(140), np.log10(300))
    xticks = [280, 240, 200, 160]
    ax.set_xticks(np.log10(xticks))
    # # Second x-axis
    ax2 = ax.twiny()
    ax2.set_xlim(ax.get_xlim())
    ax2.set_xticks(np.log10(xticks))
    ax2.set_xticklabels(xticks)
    ax2.set_xlabel(r'$T$ [K]')
    ax.legend(loc="upper right", title=r'$x$', ncol=1, bbox_to_anchor=(.48, 1.05, 0, 0),
              frameon=False, handletextpad=0.1)
    ax.tick_params(axis='both', which='both', direction='in', pad=5)
    ax2.tick_params(axis='both', which='both', direction='in', pad=5)
    ax.xaxis.set_major_formatter('{x:.2f}')
    # plt.show()


if __name__ == '__main__':
    conductivity_ht()